# 9: Using State-of-the-Art AI Stereo Matching

During this project, we have built a 3D vision pipeline that goes from stereo images
to point clouds. We have seen how we can find disparity through classical stereo
matching techniques. In this notebook, we will try out the
[S2M2](./../oaf_vision_3d/s2m2_implementation/s2m2) model for stereo matching.
S2M2 is a state-of-the-art, fully AI, stereo matching model that is at currently the
best performing stereo matching model on the
[Middlebury Stereo Evaluation](https://vision.middlebury.edu/stereo/eval/) website.

The S2M2 model is under [Creative Commons Attribution-NonCommercial 4.0 International](./../oaf_vision_3d/s2m2_implementation/LICENSE.md)
license and is available for non-commercial use only.

## Load data

We start by loading the data and visualize the stereo images. We will use the same
dataset as we used in [session 6](./../workshops/06_stereo_matching_fundamentals.ipynb).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from oaf_vision_3d._stereo_data_reader import StereoData
from oaf_vision_3d._test_data_paths import TestDataPaths
from oaf_vision_3d.point_cloud_visualization import open3d_visualize_point_cloud
from oaf_vision_3d.s2m2_implementation.s2m2 import S2M2Config, infer, load_model
from oaf_vision_3d.triangulation import triangulate_disparity

In [ ]:
data = StereoData.from_path(TestDataPaths.traproom1_dir)
image_0 = (255 * data.image_0).astype(np.uint8)
image_1 = (255 * data.image_1).astype(np.uint8)

fig, ax = plt.subplots(1, 2, figsize=(10, 6))
ax[0].imshow(image_0, cmap="gray", vmin=0, vmax=1)
ax[0].set_title("Image left")
ax[0].axis("off")
ax[1].imshow(image_1, cmap="gray", vmin=0, vmax=1)
ax[1].set_title("Image right")
ax[1].axis("off")
plt.tight_layout()
plt.show()

## Run S2M2

We can load the S2M2 model and run it on the stereo images. See the
[S2M2](./../oaf_vision_3d/s2m2_implementation/s2m2) documentation and
[Predetrained Models](./../oaf_vision_3d/s2m2_implementation/pretrained_paths) for more details.

In [ ]:
model = load_model(model_config=S2M2Config.small())

In [ ]:
disparity, occ, conf, mask = infer(model=model, left=image_0, right=image_1)

plt.figure()
plt.imshow(disparity)
plt.axis("off")
plt.colorbar()
plt.tight_layout()
plt.show()

## Triangulate disparity

We can now triangulate the disparity to get a depth map and show the depth map and
the point cloud.

In [ ]:
xyz_new = triangulate_disparity(
    disparity=disparity,
    lens_model_0=data.lens_model_0,
    lens_model_1=data.lens_model_1,
    transformation_matrix=data.transformation_matrix,
)

if xyz_new is not None:
    plt.figure(figsize=(10, 6))
    plt.imshow(xyz_new[:-32, ..., 2])
    plt.colorbar()
    plt.title("Depth Map")
    plt.axis("off")
    plt.show()

In [ ]:
if xyz_new is not None:
    open3d_visualize_point_cloud(xyz=xyz_new, rgb=data.image_0)